In [6]:
#google drive의 COSE362-term-project/dataset 폴더와 연결
from google.colab import drive

drive.mount('/content/drive')

!ls /content/drive/MyDrive/COSE362-term-project/dataset

import sys

sys.path.append('/content/drive/MyDrive/COSE362-term-project/dataset')

import os

os.chdir("/content/drive/MyDrive/COSE362-term-project/dataset")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
baseline  crop	landuse  livestock  Shapefiles


'COSE362-term-project (1)'
/content/drive/MyDrive/COSE362-term-project (1)/dataset
/content/drive/MyDrive/COSE362-term-project (1)/dataset


In [7]:
!ls

baseline  crop	landuse  livestock  Shapefiles


In [3]:
!pip install rasterstats

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 69.2 MB/s eta 0:00:00


# **LIVESTOCK DATASET Preprocessing**

## dd

In [4]:
import pandas as pd
import geopandas as gpd
from rasterstats import zonal_stats
import glob
import os
import time

print("--- 1. 기준 지도(Shapefile) 로드 ---")
# '마스터 키'가 될 Shapefile을 한 번만 불러옵니다.
shp_path = "../final_dataset/africa_shapefile/afr_g2014_2013_1.shp"
regions = gpd.read_file(shp_path)

# ADM1_CODE와 함께, 최종 테이블에 필요한 '이름' 컬럼들을 미리 준비합니다.
# (geometry는 무거우므로 제외하고, 중복을 제거합니다.)
# (ADM0_CODE은 country_code, ADM1_NAME은 admin_1으로 가정합니다.)
lookup_df = regions[['ADM0_CODE', 'ADM0_NAME', 'ADM1_CODE', 'ADM1_NAME']].drop_duplicates()
print(f"'{shp_path}'에서 총 {len(regions)}개의 지역(L1)과 {len(lookup_df)}개의 고유 ID를 로드했습니다.")


print("\n--- 2. 처리할 TIF 파일 검색 ---")
tif_folder = "livestock/africa_livestock/"
tif_files = glob.glob(f"{tif_folder}*.tif")

if not tif_files:
    print(f"🚨 [에러] '{tif_folder}' 폴더에서 .tif 파일을 찾지 못했습니다.")
else:
    print(f"총 {len(tif_files)}개의 TIF 파일을 찾았습니다.")

    all_results_list = []
    start_time = time.time()

    # --- 3. 모든 TIF 파일에 대해 Zonal Statistics 반복 실행 ---
    for i, tif_file in enumerate(tif_files, 1):

        filename = os.path.basename(tif_file)
        try:
            livestock_type, year_ext = filename.split('_')
            year = int(os.path.splitext(year_ext)[0])
        except ValueError:
            print(f"   [경고] '{filename}' 파일 이름 형식이 다릅니다. 건너뜁니다.")
            continue

        print(f"   ({i}/{len(tif_files)}) 처리 중: {livestock_type} / {year}년...")

        # Zonal Statistics 실행
        stats = zonal_stats(regions, tif_file, stats="mean")

        # ADM1_CODE와 결과(mean)를 매칭하여 리스트에 추가
        for j, s in enumerate(stats):
            mean_density = s['mean'] if s and s['mean'] is not None else 0
            gid = regions.iloc[j]['ADM1_CODE']

            all_results_list.append({
                'ADM1_CODE': gid,
                'Year': year,
                'Livestock': livestock_type,
                'Mean_Density': mean_density
            })

    print(f"\n--- 4. Zonal Statistics 완료 (총 소요 시간: {time.time() - start_time:.2f}초) ---")

    # --- 5. 피벗 테이블 생성 ---

    # 5a. "Long Format" DataFrame 생성
    df_long = pd.DataFrame(all_results_list)

    print("Pivoting: Long Format -> Wide Format으로 변환 중...")

    # 5b. Pivot 실행 (index: ADM1_CODE, Year)
    df_wide = df_long.pivot_table(
        index=['ADM1_CODE', 'Year'], # 먼저 ADM1_CODE와 Year로만 피벗합니다
        columns='Livestock',
        values='Mean_Density',
        fill_value=0
    ).reset_index()

    # --- 6. (핵심) 피벗 결과와 지역 이름(Lookup) 병합 ---
    print("Merging: 피벗 테이블과 지역(Shapefile) 정보 결합 중...")

    # ADM1_CODE을 키로 사용하여 df_wide와 lookup_df를 병합합니다.
    final_df = pd.merge(
        df_wide,
        lookup_df,
        on='ADM1_CODE',
        how='left' # df_wide(데이터)를 기준으로, lookup(이름) 정보를 왼쪽에 붙임
    )

    # --- 7. 최종 컬럼 정리 및 저장 ---

    # 7a. 요청하신 순서대로 컬럼 목록 재정의
    # (가축 컬럼 이름은 피벗 과정에서 자동으로 생성됨)
    livestock_cols = [col for col in df_wide.columns if col not in ['ADM1_CODE', 'Year']]

    # 요청하신 컬럼 순서 + 가축 컬럼
    final_columns = [
        'ADM0_CODE',
        'ADM0_NAME',
        'ADM1_CODE',
        'ADM1_NAME',
        'Year'
    ] + livestock_cols

    # 7b. 최종 DataFrame 생성 및 이름 변경
    final_df = final_df[final_columns].rename(columns={
        'Year': 'year'
    })

    # 7c. 최종 CSV 파일로 저장
    output_csv = "../final_dataset/processed_livestock.csv"
    final_df.to_csv(output_csv, index=False)

    print(f"\n--- 6. ⭐️ 성공! 최종 피처 테이블 저장 완료 ---")
    print(f"파일 위치: {output_csv}")
    print("최종 테이블 미리보기 (상위 5줄):")
    print(final_df.head())


--- 1. 기준 지도(Shapefile) 로드 ---
'../final_dataset/africa_shapefile/afr_g2014_2013_1.shp'에서 총 859개의 지역(L1)과 859개의 고유 ID를 로드했습니다.

--- 2. 처리할 TIF 파일 검색 ---
총 168개의 TIF 파일을 찾았습니다.
   (1/168) 처리 중: Horse / 2015년...


/usr/local/lib/python3.12/dist-packages/rasterstats/io.py:335: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


   (2/168) 처리 중: Buffa / 2018년...
   (3/168) 처리 중: Swine / 2019년...
   (4/168) 처리 중: Horse / 2016년...
   (5/168) 처리 중: Horse / 2014년...
   (6/168) 처리 중: Horse / 2001년...
   (7/168) 처리 중: Buffa / 2019년...
   (8/168) 처리 중: Horse / 2002년...
   (9/168) 처리 중: Horse / 2007년...
   (10/168) 처리 중: Swine / 2008년...
   (11/168) 처리 중: Swine / 2020년...
   (12/168) 처리 중: Horse / 2003년...
   (13/168) 처리 중: Horse / 2017년...
   (14/168) 처리 중: Swine / 2018년...
   (15/168) 처리 중: Horse / 2006년...
   (16/168) 처리 중: Buffa / 2009년...
   (17/168) 처리 중: Swine / 2021년...
   (18/168) 처리 중: Horse / 2013년...
   (19/168) 처리 중: Swine / 2009년...
   (20/168) 처리 중: Horse / 2012년...
   (21/168) 처리 중: Horse / 2005년...
   (22/168) 처리 중: Buffa / 2021년...
   (23/168) 처리 중: Buffa / 2020년...
   (24/168) 처리 중: Horse / 2010년...
   (25/168) 처리 중: Horse / 2011년...
   (26/168) 처리 중: Horse / 2004년...
   (27/168) 처리 중: Buffa / 2008년...
   (28/168) 처리 중: Goats / 2004년...
   (29/168) 처리 중: Chick / 2019년...
   (30/168) 처리 중: Goats / 20

In [11]:
livestock_df = pd.read_csv("../final_dataset/processed_livestock.csv")
disease_df = pd.read_csv("../final_dataset/processed_HIV_rates.csv")
country_list = set(disease_df['country'])
livestock_country_set = set(livestock_df['ADM0_NAME'])

In [12]:
delete_country = livestock_country_set - country_list
list(delete_country)

['Mauritania',
 'Morocco',
 'Equatorial Guinea',
 "CÃ´te d'Ivoire",
 'Djibouti',
 'Algeria',
 'Libya',
 'Uganda',
 'Egypt',
 'Abyei',
 'Central African Republic',
 'Tunisia',
 'Liberia',
 'Somalia',
 'Comoros',
 'Ilemi triangle',
 'South Sudan',
 'Guinea-Bissau',
 'United Republic of Tanzania',
 'Cape Verde',
 "Hala'ib triangle",
 'Madagascar',
 'Mauritius',
 'Seychelles',
 'Eritrea',
 'Botswana',
 "Ma'tan al-Sarra",
 'Sudan',
 'Nigeria',
 'Western Sahara',
 'Benin']

In [14]:
delete_condition = livestock_df['ADM0_NAME'].isin(delete_country)
livestock_df = livestock_df[~delete_condition]

In [16]:
country_list = set(disease_df['country'])
livestock_country_set = set(livestock_df['ADM0_NAME'])
livestock_country_set-country_list

set()

In [21]:
# 최종 CSV 파일로 저장
output_csv = "../final_dataset/processed_livestock.csv"
livestock_df.to_csv(output_csv, index=False)

print(f"\n--- 6. ⭐️ 성공! 최종 피처 테이블 저장 완료 ---")
print(f"파일 위치: {output_csv}")
print("최종 테이블 미리보기 (상위 5줄):")
print(livestock_df.head())



--- 6. ⭐️ 성공! 최종 피처 테이블 저장 완료 ---
파일 위치: ../final_dataset/processed_livestock.csv
최종 테이블 미리보기 (상위 5줄):
      ADM0_CODE ADM0_NAME  ADM1_CODE ADM1_NAME  year  Buffa     Cattl  \
1029          8    Angola        398     Bengo  2001    0.0  0.352811   
1030          8    Angola        398     Bengo  2002    0.0  0.342582   
1031          8    Angola        398     Bengo  2003    0.0  0.334268   
1032          8    Angola        398     Bengo  2004    0.0  0.325620   
1033          8    Angola        398     Bengo  2005    0.0  0.360941   

          Chick  Ducks     Goats     Horse     Sheep     Swine  
1029  16.751916    0.0  1.945433  0.001210  0.350924  0.578892  
1030  17.010857    0.0  2.113258  0.000992  0.394502  0.651813  
1031  17.242436    0.0  2.306114  0.001749  0.456317  0.727884  
1032  17.555852    0.0  2.398938  0.002005  0.518402  0.813876  
1033  17.131548    0.0  2.593782  0.001683  0.611763  0.807754  


In [23]:
len(set(disease_df['country']))

28